# Clasificación de Letras Escritas a Mano con Perceptrón Multicapa (MLP)

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de esta sesión es diseñar, entrenar y evaluar una red neuronal clásica basada en la arquitectura de **Perceptrón Multicapa (MLP)** para clasificar de manera automática imágenes de letras escritas a mano (del conjunto de datos EMNIST). Analizaremos el impacto de las capas densas, comprenderemos el preprocesamiento de imágenes matriciales y aprenderemos a interpretar los errores del modelo a través de matrices de confusión.

## Resultados de aprendizaje

Al final de este notebook van a poder:
1. Cargar y estructurar conjuntos de datos complejos utilizando TensorFlow Datasets (TFDS).
2. Aplicar operaciones de normalización y transformación de etiquetas en imágenes 2D.
3. Construir una arquitectura secuencial clásica con capas de aplanado (Flatten) y múltiples capas densas.
4. Evaluar cuantitativamente el desempeño de la red en datos inéditos y graficar curvas de aprendizaje.
5. Diagnosticar patrones de confusión sistemáticos en clasificación de imágenes usando matrices de error.

## Terminología clave (Microglosario)

*   **Clasificación de Imágenes:** Tarea de asignar una etiqueta categórica a una imagen basándose en sus características visuales.
*   **EMNIST (Extended MNIST):** Conjunto de datos académico que contiene imágenes digitalizadas en escala de grises de letras y dígitos escritos a mano.
*   **Aplanado (Flatten):** Proceso de transformar un tensor multidimensional (como una imagen de $28 \times 28$ píxeles) en un único vector continuo unidimensional (de $784$ elementos).
*   **Capa Oculta Densa (Dense Hidden Layer):** Capa intermedia en la cual todas sus neuronas están completamente conectadas con la salida de la capa anterior.
*   **Softmax:** Función de activación aplicada en la capa de salida para convertir las predicciones crudas de la red en un vector de probabilidades que suman exactamente 1.
*   **Matriz de Confusión:** Gráfico bidimensional que confronta la clase real de las muestras con la clase predicha por la red, permitiendo identificar errores recurrentes de clasificación.

## 1. Configuración de Dependencias e Importaciones

Comencemos instalando y cargando las herramientas de análisis de datos, visualización y desarrollo de redes neuronales.

In [ ]:
print("✦ Instalando librerías complementarias en el sistema...")
# Mantener la instalación silenciosa para evitar ruido didáctico
!pip install opencv-python-headless seaborn -q
print("✓ Librerías instaladas con éxito.\n")

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import math
import cv2
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("✓ Herramientas de Deep Learning listas para utilizar.")

## 2. Carga y Exploración de Datos (EMNIST/Letters)

Utilizaremos el dataset EMNIST configurado para la división de letras. Este conjunto agrupa miles de ejemplos de caracteres escritos a mano en escala de grises.

In [ ]:
print("✦ Descargando y cargando el dataset EMNIST/Letters en memoria...")
print("(Este proceso puede demorar unos minutos la primera vez)")

# Descargamos los datos y metadatos estructurados
datos, metadatos = tfds.load('emnist/letters', as_supervised=True, with_info=True)
print("✓ Dataset EMNIST/Letters cargado correctamente.\n")

# Generamos la lista de caracteres reales de forma explícita
nombres_clases = []
for i in range(1, 27):
    letra = chr(i + ord('a') - 1)
    nombres_clases.append(letra)

print("✓ Mapeo de clases (letras de la 'a' a la 'z'):")
print(nombres_clases)

## 3. Preprocesamiento Didáctico de las Imágenes

Los píxeles de las imágenes de entrada oscilan originalmente entre 0 y 255. Para facilitar la convergencia del entrenamiento de la red, realizaremos dos transformaciones críticas:
1. **Normalización:** Convertir los píxeles a punto flotante en el rango [0, 1].
2. **Ajuste de Etiquetas:** Dado que las etiquetas originales comienzan en 1, les restaremos 1 para adaptarlas al rango de indexación estándar de Python (0 a 25).

In [ ]:
def preprocesar_imagen(imagen, etiqueta):
    # Convertimos los valores a flotante y escalamos al rango [0, 1]
    imagen_flotante = tf.cast(imagen, tf.float32)
    imagen_normalizada = imagen_flotante / 255.0
    
    # Desplazamos la etiqueta para iniciar el índice en 0
    etiqueta_ajustada = etiqueta - 1
    
    return imagen_normalizada, etiqueta_ajustada

# Aplicamos el mapeo didáctico a los subconjuntos correspondientes
datos_entrenamiento = datos['train'].map(preprocesar_imagen)
datos_entrenamiento = datos_entrenamiento.cache()

datos_pruebas = datos['test'].map(preprocesar_imagen)
datos_pruebas = datos_pruebas.cache()

print("✓ Operaciones de preprocesamiento aplicadas y almacenadas en caché.")

## 4. Visualización de Muestras del Dataset

Es fundamental verificar visualmente la apariencia y consistencia de las imágenes procesadas antes de suministrarlas al modelo.

In [ ]:
# Extraemos un único ejemplo de entrenamiento para validación visual
for imagen, etiqueta in datos_entrenamiento.take(1):
    imagen_arreglo = imagen.numpy()
    etiqueta_numero = etiqueta.numpy()

# Configuramos y dibujamos el gráfico en escala de grises
plt.figure(figsize=(4, 4))
plt.imshow(imagen_arreglo[:, :, 0], cmap=plt.cm.binary)
plt.title(f"Ejemplo - Letra Real: '{nombres_clases[etiqueta_numero]}'", fontsize=12, fontweight="bold")
plt.colorbar(label="Intensidad Normalizada")
plt.grid(False)
plt.show()

## 5. Diseño y Compilación del Modelo de Red Neuronal (MLP)

Construiremos un Perceptrón Multicapa (MLP). Este modelo toma la matriz 2D de píxeles, la aplana para procesarla como un vector y la proyecta a través de capas ocultas completamente conectadas (Dense).

In [ ]:
# Construimos secuencialmente las capas del modelo
modelo = tf.keras.Sequential([
    # Capa de Aplanado: Redimensiona de 28x28x1 a un vector plano de 784
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    
    # Primera capa densa intermedia con activación ReLU
    tf.keras.layers.Dense(64, activation='relu'),
    
    # Segunda capa densa intermedia con activación ReLU
    tf.keras.layers.Dense(64, activation='relu'),
    
    # Capa de salida: una neurona por letra, activada por Softmax
    tf.keras.layers.Dense(len(nombres_clases), activation='softmax')
])

# Configuración del aprendizaje y las métricas de monitoreo
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Modelo secuencial de clasificación diseñado y compilado.")
modelo.summary()

## Consigna de Lectura e Interpretación

Examinen el tamaño y el número de parámetros de la primera capa oculta (`Dense` de 64 neuronas). ¿Pueden deducir de qué manera se calcula la cantidad de parámetros para esa capa basándose en el vector de entrada aplanado de 784 elementos?

## 6. Configuración de Lotes y Mezcla de Datos

Para realizar un entrenamiento eficiente, organizaremos los ejemplos de entrenamiento en pequeños grupos (lotes o batches). Adicionalmente, barajaremos el conjunto para evitar sesgos por ordenación secuencial.

In [ ]:
TAMANO_LOTE = 32
total_muestras_entrenamiento = metadatos.splits['train'].num_examples

# Dividimos y mezclamos secuencialmente los datos de entrenamiento
datos_entrenamiento_lotes = datos_entrenamiento.shuffle(total_muestras_entrenamiento)
datos_entrenamiento_lotes = datos_entrenamiento_lotes.batch(TAMANO_LOTE)
datos_entrenamiento_lotes = datos_entrenamiento_lotes.repeat()

# Los datos de prueba solo requieren agruparse para validación
datos_pruebas_lotes = datos_pruebas.batch(TAMANO_LOTE)

print(f"✓ Datos estructurados en lotes de {TAMANO_LOTE} muestras.")
print(f"  • Total de ejemplos de entrenamiento: {total_muestras_entrenamiento}")

## 7. Entrenamiento de la Red Neuronal

Alimentaremos al modelo con los lotes preparados para ajustar sus parámetros durante 15 épocas completas.

In [ ]:
EPOCAS = 15
pasos_por_epoca = math.ceil(total_muestras_entrenamiento / TAMANO_LOTE)

print(f"✦ Iniciando optimización por {EPOCAS} épocas...")
print("  (Observen cómo el error disminuye y la precisión mejora paso a paso)\n")

historial = modelo.fit(
    datos_entrenamiento_lotes,
    epochs=EPOCAS,
    steps_per_epoch=pasos_por_epoca
)

print("\n✓ Proceso de entrenamiento finalizado correctamente.")

## 8. Evaluación Cuantitativa del Modelo

Validaremos la precisión global del clasificador con datos del conjunto de pruebas que no fueron vistos durante la etapa de ajuste.

In [ ]:
print("✦ Evaluando la capacidad de generalización en muestras inéditas...")
perdida_prueba, precision_prueba = modelo.evaluate(datos_pruebas_lotes, verbose=0)

print(f"\nResultados de la Evaluación de Prueba:")
print(f"  • Pérdida (Loss) calculada: {perdida_prueba:.4f}")
print(f"  • Precisión (Accuracy) calculada: {precision_prueba:.4f} ({precision_prueba * 100:.2f}%)")

if precision_prueba > 0.80:
    print("\n✓ Diagnóstico: Excelente rendimiento y generalización. Sólido aprendizaje. ✓")
elif precision_prueba > 0.65:
    print("\n✦ Diagnóstico: Rendimiento moderado. Muestra captación básica de patrones estructurales.")
else:
    print("\n✗ Diagnóstico: Aprendizaje insuficiente. Sugiere refinar arquitectura o extender entrenamiento.")

## 9. Visualización de las Curvas de Aprendizaje

Analizaremos las curvas históricas de precisión y pérdida recolectadas durante el entrenamiento para verificar la estabilidad de la optimización.

In [ ]:
historial_dict = historial.history
rango_epocas = range(1, EPOCAS + 1)

plt.figure(figsize=(12, 5))

# Curva de Precisión
plt.subplot(1, 2, 1)
plt.plot(rango_epocas, historial_dict['accuracy'], 'bo-', label='Precisión de Entrenamiento')
plt.title('Evolución de la Precisión', fontsize=12, fontweight="bold")
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.xticks(rango_epocas)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

# Curva de Pérdida
plt.subplot(1, 2, 2)
plt.plot(rango_epocas, historial_dict['loss'], 'gs-', label='Pérdida de Entrenamiento')
plt.title('Evolución de la Pérdida', fontsize=12, fontweight="bold")
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.xticks(rango_epocas)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 10. Inspección Visual de Predicciones Individuales

Graficaremos un conjunto aleatorio de muestras con sus respectivas predicciones asignadas por el modelo, indicando de forma explícita el nivel de certeza.

In [ ]:
# Tomamos un primer lote del conjunto de validación
for imagenes_lote, etiquetas_lote in datos_pruebas_lotes.take(1):
    imagenes_arreglo = imagenes_lote.numpy()
    etiquetas_verdaderas = etiquetas_lote.numpy()
    
    # Generamos predicciones probabilisticas
    probabilidades_lote = modelo.predict(imagenes_arreglo)
    break

def graficar_muestra_prediccion(indice, probabilidades, reales, imagenes):
    distribucion = probabilidades[indice]
    clase_real = reales[indice]
    imagen_matriz = imagenes[indice]
    
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(imagen_matriz[:, :, 0], cmap=plt.cm.binary)
    
    clase_predicha = np.argmax(distribucion)
    porcentaje_confianza = np.max(distribucion) * 100
    
    nombre_real = nombres_clases[clase_real]
    nombre_prediccion = nombres_clases[clase_predicha]
    
    # Color azul para acierto y rojo para falla de clasificación
    color_etiqueta = 'blue' if clase_predicha == clase_real else 'red'
    plt.xlabel(f"Predijo: {nombre_prediccion} ({porcentaje_confianza:.0f}%)\nReal: {nombre_real}", color=color_etiqueta)

filas_grafico = 3
columnas_grafico = 5
total_imagenes_mostrar = filas_grafico * columnas_grafico

plt.figure(figsize=(2 * columnas_grafico + 1, 2 * filas_grafico + 2))
plt.suptitle("Muestra de Inferencias (Azul: Acierto, Rojo: Error)", fontsize=14, fontweight="bold")

for i in range(total_imagenes_mostrar):
    plt.subplot(filas_grafico, columnas_grafico, i + 1)
    graficar_muestra_prediccion(i, probabilidades_lote, etiquetas_verdaderas, imagenes_arreglo)
    
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 11. Diagnóstico de Errores con Matriz de Confusión

Para comprender en qué caracteres se generan confusiones recurrentes, construiremos y graficaremos la matriz de confusión sobre la totalidad de las muestras de prueba.

In [ ]:
print("✦ Recopilando inferencias completas del conjunto de prueba...")
print("  (Por favor, aguarden un instante)")

acumulado_predicciones = []
acumulado_verdades = []

for imagenes, etiquetas in datos_pruebas_lotes:
    predicciones_lote = modelo.predict(imagenes, verbose=0)
    clases_lote = np.argmax(predicciones_lote, axis=1)
    
    acumulado_predicciones.extend(clases_lote)
    acumulado_verdades.extend(etiquetas.numpy())

vector_predicciones = np.array(acumulado_predicciones)
vector_verdades = np.array(acumulado_verdades)

# Calculamos la matriz de error
matriz_error = confusion_matrix(vector_verdades, vector_predicciones)

# Diseñamos el mapa de calor con Seaborn
plt.figure(figsize=(12, 10))
sns.heatmap(
    matriz_error, 
    annot=False, 
    cmap='Blues', 
    xticklabels=nombres_clases, 
    yticklabels=nombres_clases
)
plt.xlabel('Predicción del Modelo', fontsize=12, fontweight="bold")
plt.ylabel('Etiqueta Real', fontsize=12, fontweight="bold")
plt.title('Matriz de Confusión: Distribución de Errores por Clase', fontsize=14, fontweight="bold", pad=20)
plt.show()

print("✦ Guía de Análisis e Interpretación:\n")
print("  • La diagonal principal muestra los aciertos acumulados para cada carácter.")
print("  • Los puntos dispersos por fuera de la diagonal representan desviaciones sistemáticas.")
print("  • Por ejemplo, busquen las posiciones correspondientes a letras similares (como 'i' y 'l' u 'o' y 'q').")

## Consigna de Lectura e Interpretación

Analicen detenidamente las desviaciones visuales de la matriz. ¿Cuáles son los pares de letras que sufren la mayor tasa de confusión cruzada? ¿A qué creen que se debe esta confusión visual basándose en la forma estructural de las letras escritas a mano?

## Cierre de Laboratorio

En esta práctica estructuramos un flujo de aprendizaje automático para imágenes digitalizadas de letras escritas a mano utilizando un Perceptrón Multicapa (MLP). Comprendimos que aplanar las imágenes remueve su estructura espacial bidimensional básica, lo cual restringe notablemente la robustez del modelo ante rotaciones, desplazamientos y deformaciones del carácter.

En la siguiente sesión daremos el gran salto hacia las **Redes Neuronales Convolucionales (CNNs)**, diseñadas específicamente para preservar la topología de la imagen mediante kernels de convolución.